In [ ]:
# Parameters — injected by CI (do not edit manually)
ci_target = "ephemeral_ci"
prod_state_path = "abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/Files/ci-artifacts/prod-state"
repo_url = "https://github.com/ORG/REPO.git"
repo_branch = "BRANCH"
github_app_id = "GH-APP-ID"
github_installation_id = "GH-APP-INSTALLATION-ID"
github_pem_secret = "GH-APP-PEM"
vault_url = "https://KV-NAME.vault.azure.net/"
lakehouse_name = "LAKEHOUSE_NAME"
lakehouse_id = "LAKEHOUSE_ID"
workspace_id = "WORKSPACE_ID"
workspace_name = "WORKSPACE_NAME"
schema_name = "dbo"
gate = "2"
head_sha = "HEAD_SHA"

In [ ]:
# Download prod-state manifest from OneLake (ABFSS → local path)
# No-op when prod_state_path is already a local path (e.g. greenfield ./prod-state).
if prod_state_path.startswith('abfss://'):
    import os
    # The provision job uploads the prod-state manifest to the ephemeral lakehouse
    # at Files/ci-artifacts/prod-state/manifest.json before this notebook runs.
    # Use the FUSE-mounted path instead of /tmp/ — /tmp/ on the Jupyter driver is
    # not visible inside run_dbt_job's execution context (Livy subprocess isolation).
    local_dir = '/lakehouse/default/Files/ci-artifacts/prod-state'
    manifest_file = f'{local_dir}/manifest.json'
    print(f"Checking prod-state manifest at: {manifest_file}")
    if not os.path.exists(manifest_file):
        raise RuntimeError(
            f"Prod-state manifest not found at {manifest_file}. "
            f"Expected to be uploaded by the provision job (source: {prod_state_path})."
        )
    local_prod_state_path = local_dir
    print(f"Prod-state manifest verified at {manifest_file}")
    # prod-state-defer/manifest.json holds the prod-target manifest for --defer
    # relation resolution. Falls back to compile manifest if not uploaded (VD-2142).
    _defer_dir = '/lakehouse/default/Files/ci-artifacts/prod-state-defer'
    _defer_file = f'{_defer_dir}/manifest.json'
    local_prod_defer_path = _defer_dir if os.path.exists(_defer_file) else local_prod_state_path
else:
    local_prod_state_path = prod_state_path
    local_prod_defer_path = prod_state_path

In [ ]:
# Command assembly — common commands
_deps = "dbt deps"

# Gate 2, Gate 3, and Gate 4 commands are assembled dynamically in check_gate_2 / check_gate_3 / check_gate_4
# using explicit model names from the deployment manifest (VD-2142, VD-2175).


In [ ]:
!pip install vd-dbt-fabricspark==1.9.15 -q

In [ ]:
from typing import Any, Callable

ExecutorFn = Callable[[str], list]

ColumnInfo = dict  # {"name": str, "dtype": str, "nullable": bool}

VALUE_DELTA_MATERIALIZATIONS: frozenset[str] = frozenset({"table", "incremental", "snapshot"})
SKIP_MATERIALIZATIONS: frozenset[str] = frozenset({"ephemeral"})


# ── Pure core ──────────────────────────────────────────────────────────────────

def compute_schema_delta(
    prod_columns: list[ColumnInfo],
    ci_columns: list[ColumnInfo],
) -> dict:
    """Return schema delta: added, removed, renamed, type_changed, nullability_flipped."""
    prod_by_name = {c["name"]: c for c in prod_columns}
    ci_by_name = {c["name"]: c for c in ci_columns}
    prod_names = set(prod_by_name)
    ci_names = set(ci_by_name)

    type_changed = []
    nullability_flipped = []
    for name in sorted(prod_names & ci_names):
        p, c = prod_by_name[name], ci_by_name[name]
        if p.get("dtype") != c.get("dtype"):
            type_changed.append({
                "column": name,
                "prod_dtype": p.get("dtype"),
                "ci_dtype": c.get("dtype"),
            })
        if bool(p.get("nullable")) != bool(c.get("nullable")):
            nullability_flipped.append({
                "column": name,
                "prod_nullable": p.get("nullable"),
                "ci_nullable": c.get("nullable"),
            })

    return {
        "added": sorted(ci_names - prod_names),
        "removed": sorted(prod_names - ci_names),
        "renamed": [],
        "type_changed": type_changed,
        "nullability_flipped": nullability_flipped,
    }


def compute_row_count_delta(prod_count: int, ci_count: int) -> dict:
    """Return {"prod": int, "pr": int, "delta": int}. delta = pr - prod."""
    return {"prod": prod_count, "pr": ci_count, "delta": ci_count - prod_count}


def normalize_unique_key(raw: Any) -> list[str] | None:
    """Normalise manifest unique_key to list[str] or None.

    manifest.json stores unique_key as a string (single-column) or list (composite).
    Returns None when key is absent, empty, or an unexpected type.
    """
    if raw is None:
        return None
    if isinstance(raw, str):
        return [raw] if raw else None
    if isinstance(raw, list):
        return raw if raw else None
    return None


def _diff_where(pa: str, ca: str, keys: list[str], common_columns: list[str]) -> str:
    """Generate WHERE clause to detect differing rows via FULL OUTER JOIN.

    Detects:
    - Rows added to CI (missing in prod, detected by prod key IS NULL)
    - Rows deleted from prod (missing in CI, detected by CI key IS NULL)
    - Rows with value changes in non-key columns (detected by IS DISTINCT FROM)
    """
    non_key = [c for c in common_columns if c not in set(keys)]
    null_side = f"({pa}.`{keys[0]}` IS NULL OR {ca}.`{keys[0]}` IS NULL)"
    if not non_key:
        return null_side
    col_diffs = " OR ".join(
        f"({pa}.`{c}` IS DISTINCT FROM {ca}.`{c}`)" for c in non_key
    )
    return f"({null_side} OR ({col_diffs}))"


def build_value_delta_count_sql(
    prod_table: str,
    ci_table: str,
    keys: list[str],
    common_columns: list[str],
) -> str:
    """Return a COUNT query that counts differing rows via key-based FULL OUTER JOIN.

    Joins prod and CI tables on all key columns, then counts rows where:
    - Either side is NULL (row added/deleted), or
    - Non-key column values differ (IS DISTINCT FROM).
    """
    join_on = " AND ".join(f"p.`{k}` = c.`{k}`" for k in keys)
    where = _diff_where("p", "c", keys, common_columns)
    return (
        f"SELECT COUNT(*) AS rows_with_diffs "
        f"FROM {prod_table} p "
        f"FULL OUTER JOIN {ci_table} c ON {join_on} "
        f"WHERE {where}"
    )


# ── I/O shell ─────────────────────────────────────────────────────────────────

def _extract_scalar(result: list) -> int:
    """Extract a single integer from a one-row executor result.

    Handles both dict rows ({"rows_with_diffs": n}) and tuple rows ((n,)).
    """
    if not result:
        return 0
    row = result[0]
    if isinstance(row, dict):
        return int(next(iter(row.values())))
    if isinstance(row, (list, tuple)):
        return int(row[0])
    return int(row)


def execute_value_delta(
    executor: "ExecutorFn",
    prod_table: str,
    ci_table: str,
    keys: list[str],
    common_columns: list[str],
) -> dict:
    """Run the key-based JOIN count query via executor; return value_delta dict.

    executor(sql) must return a list with one row containing the count.
    Use a mocked executor in tests; pass a real Spark SQL executor in the notebook.
    """
    sql = build_value_delta_count_sql(prod_table, ci_table, keys, common_columns)
    rows_with_diffs = _extract_scalar(executor(sql))
    return {
        "sampled_rows": min(rows_with_diffs, 10000),
        "rows_with_diffs": rows_with_diffs,
        "skipped_no_unique_key": False,
    }


# ── Artifact-level orchestration ──────────────────────────────────────────────

def compute_artifact_diff(
    artifact: dict,
    prod_columns: list[ColumnInfo],
    ci_columns: list[ColumnInfo],
    prod_count: int,
    ci_count: int,
    value_delta: dict | None,
) -> dict:
    """Assemble the per-artifact result dict from pre-fetched values.

    The calling notebook cell (VD-1974) is responsible for:
    - skipping ephemeral materializations (check SKIP_MATERIALIZATIONS before calling)
    - deciding whether to compute value delta (check VALUE_DELTA_MATERIALIZATIONS)
    - fetching prod_columns, ci_columns, prod_count, ci_count from the database
    - passing value_delta=None for view/materialized_view (demo trim)
    - passing {"skipped_no_unique_key": True, ...} when model has no unique_key

    brand-new artifact (pre_existing_in_prod: false) → baseline: null, no deltas.
    """
    unique_id = artifact["unique_id"]
    name = artifact.get("name", "")
    materialized = artifact.get("materialized", "")

    if not artifact.get("pre_existing_in_prod", True):
        return {
            "unique_id": unique_id,
            "name": name,
            "materialized": materialized,
            "baseline": None,
            "schema_delta": None,
            "row_count_delta": None,
            "value_delta": None,
        }

    return {
        "unique_id": unique_id,
        "name": name,
        "materialized": materialized,
        "baseline": name,
        "schema_delta": compute_schema_delta(prod_columns, ci_columns),
        "row_count_delta": compute_row_count_delta(prod_count, ci_count),
        "value_delta": value_delta,
    }


# ── Gate signal + result builder ───────────────────────────────────────────────

def gate_5_overall_status(artifacts: list[dict], *, session_error: bool = False) -> str:
    """Return 'pass', 'fail', or 'error'.

    'error': session_error=True — Livy session dead, no diff computed.
    'fail': any artifact with non-empty schema, row-count, or value diff.
    'pass': all brand-new or no non-empty diff.
    Skipped value delta (skipped_no_unique_key: true or value_delta: null) does NOT fail.
    """
    if session_error:
        return "error"
    for a in artifacts:
        if a.get("baseline") is None:
            continue  # brand-new — auto-pass

        schema = a.get("schema_delta") or {}
        if any(schema.get(k) for k in ("added", "removed", "renamed", "type_changed", "nullability_flipped")):
            return "fail"

        row = a.get("row_count_delta") or {}
        if row.get("delta", 0) != 0:
            return "fail"

        value = a.get("value_delta") or {}
        if value and not value.get("skipped_no_unique_key", False) and value.get("rows_with_diffs", 0) > 0:
            return "fail"

    return "pass"


def build_gate_5_result(head_sha: str, artifacts: list[dict], *, session_error=None) -> dict:
    """Assemble the gate-5.json dict (design spec §9.2.2).

    Pass session_error=<msg> for dead-session path; overall_status becomes 'error'
    and a top-level session_error field is included.
    """
    result = {
        "gate": "5",
        "head_sha": head_sha,
        "overall_status": gate_5_overall_status(artifacts, session_error=bool(session_error)),
        "artifacts": artifacts,
    }
    if session_error:
        result["session_error"] = session_error
    return result


# ── Livy I/O shell (VD-2065) ──────────────────────────────────────────────────

import json  # noqa: E402
import time  # noqa: E402
import urllib.error  # noqa: E402
import urllib.request  # noqa: E402


def execute_livy_statement(session_url, code, token_fn, timeout_s=90):
    """Submit PySpark code to a Livy session; return text/plain output.

    Polls every 2s up to timeout_s for state == 'available'. Default 90s suits
    warm sessions; pass a larger value (e.g. 300) for the first statement after
    create_livy_session, where Spark executor cold-start adds 60–180s.
    Raises RuntimeError on output error, TimeoutError on timeout.
    """
    token = token_fn()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    body = json.dumps({"code": code, "kind": "pyspark"}).encode()
    req = urllib.request.Request(f"{session_url}/statements", data=body, method="POST", headers=headers)
    with urllib.request.urlopen(req) as resp:
        stmt = json.loads(resp.read())
    stmt_id = stmt["id"]

    stmt_url = f"{session_url}/statements/{stmt_id}"
    elapsed = 0
    while elapsed < timeout_s:
        time.sleep(2)
        elapsed += 2
        req = urllib.request.Request(stmt_url, headers={"Authorization": f"Bearer {token}"})
        with urllib.request.urlopen(req) as resp:
            stmt = json.loads(resp.read())
        if stmt["state"] == "available":
            output = stmt.get("output") or {}
            if output.get("status") == "error":
                raise RuntimeError(output.get("evalue", "Livy statement error"))
            return (output.get("data") or {}).get("text/plain", "")
    raise TimeoutError(f"Livy statement {stmt_id} did not complete within {timeout_s}s")


def execute_spark_schema(session_url, table_ref, token_fn):
    """Fetch table schema via Livy; return list of ColumnInfo dicts."""
    code = f"print(spark.table('{table_ref}').schema.json())"
    output = execute_livy_statement(session_url, code, token_fn)
    schema = json.loads(output)
    return [
        {"name": f["name"], "dtype": f["type"] if isinstance(f["type"], str) else json.dumps(f["type"]), "nullable": f["nullable"]}
        for f in schema["fields"]
    ]


def execute_spark_count(session_url, table_ref, token_fn):
    """Fetch table row count via Livy."""
    code = f"print(spark.table('{table_ref}').count())"
    return int(execute_livy_statement(session_url, code, token_fn).strip())


def make_livy_sql_executor(session_url, token_fn):
    """Return an ExecutorFn that runs SQL via Livy and returns a list of row dicts."""
    def executor(sql):
        code = f"import json\nprint(json.dumps([row.asDict() for row in spark.sql({sql!r}).collect()]))"
        output = execute_livy_statement(session_url, code, token_fn)
        return json.loads(output)
    return executor


def create_livy_session(livy_base_url, session_name, token_fn):
    """Create a new Livy session; return session_id as str.

    POSTs to {livy_base_url}/sessions, polls every 2s up to 600s for state == 'idle'.
    Raises TimeoutError on poll timeout; RuntimeError on terminal state (dead/error/killed).
    """
    tok = token_fn()
    headers = {"Authorization": f"Bearer {tok}", "Content-Type": "application/json"}
    body = json.dumps({"name": session_name}).encode()
    req = urllib.request.Request(f"{livy_base_url}/sessions", data=body, method="POST", headers=headers)
    with urllib.request.urlopen(req) as resp:
        session = json.loads(resp.read())
    session_id = session["id"]
    poll_url = f"{livy_base_url}/sessions/{session_id}"
    session_url = f"{livy_base_url}/sessions/{session_id}"

    try:
        elapsed = 0
        while elapsed < 600:
            time.sleep(2)
            elapsed += 2
            req = urllib.request.Request(poll_url, headers={"Authorization": f"Bearer {token_fn()}"})
            with urllib.request.urlopen(req) as resp:
                session = json.loads(resp.read())
            state = session.get("state")
            if state == "idle":
                return str(session_id)
            if state in ("dead", "error", "killed"):
                raise RuntimeError(f"Livy session {session_id} entered terminal state: {state}")
        raise TimeoutError(f"Livy session {session_id} did not reach idle within 600s")
    except Exception:
        delete_livy_session(session_url, token_fn)
        raise


def delete_livy_session(session_url, token_fn):
    """Delete a Livy session. Idempotent: swallows 404."""
    req = urllib.request.Request(
        session_url, method="DELETE",
        headers={"Authorization": f"Bearer {token_fn()}"},
    )
    try:
        with urllib.request.urlopen(req):
            pass
    except urllib.error.HTTPError as e:
        if e.code != 404:
            raise
    except urllib.error.URLError:
        pass


In [ ]:
from dbt.adapters.fabricspark.notebook import (
    run_dbt_job,
    DbtJobConfig,
    RepoConfig,
    ConnectionConfig,
)

In [ ]:
# CI gate functions
import json  # noqa: F811
import os
_pr = workspace_name.rsplit("_", 1)[-1] if workspace_name else "manual"
os.environ["PR_NUMBER"] = _pr
_FUSE = "/lakehouse/default/Files/ci-artifacts"
_SESSION_CONFIG = {
    "2": f"{_FUSE}/livy-session-id-ci-run.txt",
    "3": f"{_FUSE}/livy-session-id-ci-unit-test.txt",
    "4": f"{_FUSE}/livy-session-id-ci-data-test.txt",
}
if gate in _SESSION_CONFIG:
    os.environ["SESSION_ID_FILE"] = _SESSION_CONFIG[gate]

# vd-dbt-fabricspark always clones to this directory
_PROJECT_DIR = "/tmp/dbt_project"

_repo = RepoConfig(
    url=repo_url,
    branch=repo_branch,
    github_app_id=github_app_id,
    github_installation_id=github_installation_id,
    github_pem_secret=github_pem_secret,
    vault_url=vault_url,
)
_conn = ConnectionConfig(
    lakehouse_name=lakehouse_name,
    lakehouse_id=lakehouse_id,
    workspace_id=workspace_id,
    workspace_name=workspace_name,
    schema_name=schema_name,
)

def _make_job(commands):
    return DbtJobConfig(command=commands, repo=_repo, connection=_conn)

def _load_run_results(subdir):
    """Load run_results.json from the dbt project's target directory."""
    path = os.path.join(_PROJECT_DIR, subdir, "run_results.json")
    return json.load(open(path)) if os.path.exists(path) else {"results": []}

def _write_gate_result(gate_num, result_dict):
    import urllib.request
    data = json.dumps(result_dict, indent=2).encode("utf-8")
    remote = f"Files/ci-artifacts/gate-{gate_num}/{head_sha}/gate-{gate_num}.json"
    dir_remote = f"Files/ci-artifacts/gate-{gate_num}/{head_sha}"
    dfs_base = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lakehouse_id}"
    file_url = f"{dfs_base}/{remote}"
    dir_url = f"{dfs_base}/{dir_remote}"
    token = notebookutils.credentials.getToken("storage")  # noqa: F821
    auth = {"Authorization": f"Bearer {token}"}
    def _dfs(url, method, body=b"", extra=None):
        req = urllib.request.Request(url, data=body or None, method=method,
                                     headers={**auth, **(extra or {})})
        with urllib.request.urlopen(req) as r:
            return r.status
    _dfs(f"{dir_url}?resource=directory", "PUT")  # create parent dir first
    _dfs(f"{file_url}?resource=file&overwrite=true", "PUT")
    _dfs(f"{file_url}?action=append&position=0", "PATCH", data,
         {"Content-Length": str(len(data)), "Content-Type": "application/octet-stream"})
    _dfs(f"{file_url}?action=flush&position={len(data)}", "PATCH")
    print(f"Gate {gate_num} result written to OneLake: {remote}")

def _model_rows(run_results, materialization_map=None):
    _mat_map = materialization_map or {}
    return [
        {
            "name": r.get("unique_id", "").split(".")[-1],
            "status": r.get("status", ""),
            "duration_seconds": r.get("execution_time", 0.0),
            "error_message": (r.get("message") or "")[:500] or None,
            "materialization": _mat_map.get(r.get("unique_id", ""), ""),
            "rows": (r.get("adapter_response") or {}).get("rows_affected"),
        }
        for r in run_results.get("results", [])
    ]

def _step_status(models):
    return "pass" if all(m["status"] in ("success", "pass") for m in models) else "fail"


def check_gate_2():
    manifest_raw = notebookutils.fs.head(  # noqa: F821
        f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
        f"/Files/ci-artifacts/deployment-manifest-{head_sha}.json"
    )
    artifacts = json.loads(manifest_raw).get("artifacts", [])
    model_names = [
        a["name"].split(".")[-1]
        for a in artifacts
        if a.get("materialized") != "ephemeral"
    ]
    if not model_names:
        _write_gate_result("2", {
            "gate": "2", "head_sha": head_sha, "overall_status": "pass",
            "clone": {"status": "pass", "models": []},
            "build": {"status": "pass", "models": []},
        })
        return
    _names = " ".join(model_names)
    _clone_cmd = (
        f"dbt clone --select {_names} --exclude config.materialized:view"
        f" --state {local_prod_defer_path}"
        f" --target-path target/clone --profiles-dir .github/profiles --target {ci_target}"
    )
    _run_cmd = (
        f"dbt run --select {_names}"
        f" --defer --state {local_prod_defer_path}"
        f" --target-path target/build --profiles-dir .github/profiles --target {ci_target}"
    )
    gate2_cmds = [_deps, _clone_cmd, _run_cmd]
    _exc = None
    try:
        run_dbt_job(_make_job(gate2_cmds))
    except Exception as e:
        _exc = e
    _build_results_path = os.path.join(_PROJECT_DIR, "target/build/run_results.json")
    if _exc is not None and not os.path.exists(_build_results_path):
        _write_gate_result("2", {
            "gate": "2", "head_sha": head_sha, "overall_status": "fail",
            "clone": {"status": "fail", "models": []},
            "build": {"status": "fail", "models": []},
            "error": str(_exc)[:500],
        })
        return
    _manifest_path = os.path.join(_PROJECT_DIR, "target/build/manifest.json")
    _materialization_map = {}
    if os.path.exists(_manifest_path):
        with open(_manifest_path) as _f:
            _manifest_data = json.load(_f)
        for _uid, _node in (_manifest_data.get("nodes") or {}).items():
            _mat = (_node.get("config") or {}).get("materialized", "")
            if _mat:
                _materialization_map[_uid] = _mat
    clone_models = _model_rows(_load_run_results("target/clone"), _materialization_map)
    build_models = _model_rows(_load_run_results("target/build"), _materialization_map)
    clone_status = _step_status(clone_models)
    build_status = _step_status(build_models)
    overall = "pass" if clone_status == "pass" and build_status == "pass" else "fail"
    _write_gate_result("2", {
        "gate": "2", "head_sha": head_sha, "overall_status": overall,
        "clone": {"status": clone_status, "models": clone_models},
        "build": {"status": build_status, "models": build_models},
    })


def check_gate_3():
    manifest_raw = notebookutils.fs.head(  # noqa: F821
        f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/ci-artifacts/deployment-manifest-{head_sha}.json"
    )
    artifacts = json.loads(manifest_raw).get("artifacts", [])
    model_names = [
        a["name"].split(".")[-1]
        for a in artifacts
        if a.get("materialized") != "ephemeral"
    ]
    if not model_names:
        _write_gate_result("3", {
            "gate": "3", "head_sha": head_sha, "overall_status": "pass",
            "total": 0, "counts": {"pass": 0, "fail": 0, "error": 0, "skip": 0},
            "failures": [], "truncated": False, "tests": [],
        })
        return
    _unit_dynamic = (
        f"dbt test --select {' '.join(f'{n},test_type:unit' for n in model_names)}"
        f" --defer --state {local_prod_defer_path}"
        f" --target-path target/unit --profiles-dir .github/profiles --target {ci_target}"
    )
    gate3_commands_dynamic = [_deps, _unit_dynamic]
    try:
        run_dbt_job(_make_job(gate3_commands_dynamic))
    except Exception:
        pass
    import sys
    sys.path.insert(0, os.path.join(_PROJECT_DIR, ".github", "scripts"))
    from parse_run_results import summarize
    run_results = _load_run_results("target/unit")
    summary = summarize(run_results)
    _write_gate_result("3", {
        "gate": "3", "head_sha": head_sha,
        **summary,
    })


def check_gate_4():
    manifest_raw = notebookutils.fs.head(  # noqa: F821
        f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
        f"/Files/ci-artifacts/deployment-manifest-{head_sha}.json"
    )
    artifacts = json.loads(manifest_raw).get("artifacts", [])
    model_names = [
        a["name"].split(".")[-1]
        for a in artifacts
        if a.get("materialized") != "ephemeral"
    ]
    if not model_names:
        _write_gate_result("4", {
            "gate": "4", "head_sha": head_sha, "overall_status": "pass",
            "tests": [],
        })
        return
    _names = " ".join(model_names)
    _data_cmd = (
        f"dbt test --select {_names} --exclude test_type:unit --store-failures"
        f" --defer --state {local_prod_defer_path}"
        f" --target-path target/data --profiles-dir .github/profiles --target {ci_target}"
    )
    gate4_cmds = [_deps, _data_cmd]
    try:
        run_dbt_job(_make_job(gate4_cmds))
    except Exception:
        pass
    _manifest_path = os.path.join(_PROJECT_DIR, "target/data/manifest.json")
    _test_model_map = {}
    if os.path.exists(_manifest_path):
        with open(_manifest_path) as _f:
            _manifest_data = json.load(_f)
        for _uid, _node in (_manifest_data.get("nodes") or {}).items():
            if _node.get("resource_type") == "test":
                _attached = _node.get("attached_node") or _node.get("model") or ""
                if _attached:
                    _test_model_map[_uid] = _attached.split(".")[-1]
    run_results = _load_run_results("target/data")
    tests_out = [
        {
            "name": r.get("unique_id", "").split(".")[-1],
            "model": _test_model_map.get(r.get("unique_id", ""), ""),
            "status": r.get("status", ""),
            "duration_seconds": r.get("execution_time", 0.0),
            "failures_count": r.get("failures", 0) or 0,
            "store_failures_table": f"dbt_test__audit.{r.get('unique_id', '').split('.')[-1]}" if r.get("status") in ("fail", "error") else None,
            "message": (r.get("message") or "")[:500] or None,
        }
        for r in run_results.get("results", [])
    ]
    overall = "fail" if any(t["status"] in ("fail", "error") for t in tests_out) else "pass"
    _write_gate_result("4", {
        "gate": "4", "head_sha": head_sha, "overall_status": overall,
        "tests": tests_out,
    })


def check_gate_5():
    _livy_base = (
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"
        f"/lakehouses/{lakehouse_id}/livyapi/versions/2023-12-01"
    )
    token_fn = lambda: notebookutils.credentials.getToken("pbi")  # noqa: F821, E731
    session_url = None
    try:
        try:
            session_id = create_livy_session(
                _livy_base, f"dbt_ci_data_diff_{_pr}", token_fn,
            )
        except Exception as _e:
            _write_gate_result("5", build_gate_5_result(
                head_sha, [], session_error=f"Gate 5 session creation failed: {_e}",
            ))
            return
        session_url = f"{_livy_base}/sessions/{session_id}"

        # Warm-up: force Spark executor allocation on a fresh session before the
        # artifact loop. create_livy_session waits for driver idle, but executor
        # cold-start (60-180s) only happens on the first action — without this,
        # whichever artifact is first absorbs the cost and times out at 90s.
        try:
            execute_livy_statement(
                session_url,
                "print(spark.range(1).count())",
                token_fn,
                timeout_s=300,
            )
        except Exception as _e:
            _write_gate_result("5", build_gate_5_result(
                head_sha, [], session_error=f"Gate 5 session warm-up failed: {_e}",
            ))
            return

        # Read deployment manifest
        manifest_raw = notebookutils.fs.head(  # noqa: F821
            f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
            f"/Files/ci-artifacts/deployment-manifest-{head_sha}.json"
        )
        deployment_manifest = json.loads(manifest_raw)
        artifacts_in = deployment_manifest.get("artifacts", [])

        artifacts_out = []
        for artifact in artifacts_in:
            mat = artifact.get("materialized", "")
            if mat in SKIP_MATERIALIZATIONS:
                continue
            if not artifact.get("pre_existing_in_prod", True):
                artifacts_out.append(compute_artifact_diff(artifact, [], [], 0, 0, None))
                continue

            name = artifact.get("name", "")
            table_name = name.split(".")[-1]
            try:
                artifact_schema = artifact.get("schema", "dbo")
                ci_ref = f"`{lakehouse_name}`.`{artifact_schema}`.`{table_name}`"
                prod_ref = f"`{prod_workspace_name}`.`{prod_lakehouse_name}`.`{artifact_schema}`.`{table_name}`"  # noqa: F821

                ci_cols = execute_spark_schema(session_url, ci_ref, token_fn)
                prod_cols = execute_spark_schema(session_url, prod_ref, token_fn)
                ci_count = execute_spark_count(session_url, ci_ref, token_fn)
                prod_count = execute_spark_count(session_url, prod_ref, token_fn)

                value_delta = None
                if mat in VALUE_DELTA_MATERIALIZATIONS:
                    raw_key = artifact.get("unique_key")
                    keys = normalize_unique_key(raw_key)
                    if keys:
                        common = sorted(
                            {c["name"] for c in ci_cols} & {c["name"] for c in prod_cols}
                        )
                        value_delta = execute_value_delta(
                            make_livy_sql_executor(session_url, token_fn),
                            prod_ref, ci_ref, keys, common,
                        )
                    else:
                        value_delta = {
                            "sampled_rows": 0,
                            "rows_with_diffs": 0,
                            "skipped_no_unique_key": True,
                        }

                artifacts_out.append(
                    compute_artifact_diff(artifact, prod_cols, ci_cols, prod_count, ci_count, value_delta)
                )
            except Exception as e:
                artifacts_out.append({
                    "unique_id": artifact.get("unique_id", ""),
                    "name": name,
                    "materialized": mat,
                    "baseline": None,
                    "schema_delta": None,
                    "row_count_delta": None,
                    "value_delta": None,
                    "error": str(e)[:500],
                })

        _write_gate_result("5", build_gate_5_result(head_sha, artifacts_out))
    finally:
        if session_url is not None:
            try:
                delete_livy_session(session_url, token_fn)
            except Exception:
                pass


In [ ]:
# CI dispatch
gate_fns = {"2": check_gate_2, "3": check_gate_3, "4": check_gate_4, "5": check_gate_5}
gate_fns[gate]()